# SlideRule: on-demand ICESat-2 and GEDI processing

[SlideRule](https://slideruleearth.io/) processes satellite altimetry and
lidar data in the cloud. `ea.plugins.sliderule` sends a processor, its
parameters and an area of interest, and returns geolocated points as a
`GeoDataFrame`. SlideRule manages its own credentials.

## Setup

```bash
pip install earthaccess earthaccess-plugin-sliderule geopandas matplotlib
```

Register at https://slideruleearth.io and use your organization below.

In [ ]:
import earthaccess as ea

ea.login()
ea.plugins.sliderule.configure(url="slideruleearth.io", organization="sliderule")
ea.list_plugins(details=True)

## Search

A small ATL03 (ICESat-2) region over Greenland.

In [ ]:
bbox = (-43.5, 66.5, -43.3, 66.7)  # a small ATL03 region

atl03 = ea.search_data(
    short_name="ATL03",
    bounding_box=bbox,
    temporal=("2023-06-01", "2023-06-30"),
    count=1,
)
len(atl03)

## What SlideRule can do

The plugin reports the processors it can run and the subsetting it does:

In [ ]:
ea.plugins.sliderule.capabilities("ATL03")

## Process with `atl03x`

`request(...).results()` sends the request and returns a `GeoDataFrame`. The searched
granules scope the request; the `aoi` restricts the region.

In [ ]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [[-43.5, 66.5], [-43.3, 66.5], [-43.3, 66.7], [-43.5, 66.7], [-43.5, 66.5]]
    ],
}

df = ea.plugins.sliderule.request(atl03, api="atl03x", parms={}, aoi=aoi).results()
df.shape

In [ ]:
df.head()

In [ ]:
df.plot(column="height", markersize=0.5)  # geolocated points from SlideRule

## Other processors and parameters

Pass any SlideRule API and its parameters. ICESat-2 APIs include `atl03sp`,
`atl06p`, `atl06sp`, `atl08p`, `atl13sp`, `atl24v`; GEDI APIs include
`gedil1b`, `gedil2a`, `gedil4a`. `parms` is passed through unchanged:

```python
segments = ea.plugins.sliderule.request(
    atl03, api="atl06p", parms={"pass_invalid": False}, aoi=aoi,
).results()
```

You can also run from an `aoi` + `parms` alone, with no search results.

In [ ]:
segments = ea.plugins.sliderule.request(
    atl03, api="atl06p", parms={}, aoi=aoi
).results()
segments.shape

## Unsupported datasets fail early

SlideRule only handles ICESat-2 and GEDI; anything else raises before the
backend is contacted:

In [ ]:
from earthaccess.plugins import UnsupportedCollectionError

landsat = ea.search_data(short_name="LANDSAT_8_C1", count=1)
try:
    ea.plugins.sliderule.request(landsat, api="atl06p").results()
except UnsupportedCollectionError as exc:
    print(exc)

## Which plugins accept a dataset?

In [ ]:
atl03.list_services()

## Summary

- `ea.plugins.sliderule.request(results, api=..., parms=..., aoi=...).results()`
  → `GeoDataFrame`.
- SlideRule keeps its own credentials (`configure(...)`).
- Supports only ICESat-2 and GEDI collections; others fail early.